# 04. 교정, 앙상블 및 추론

Qwen raw OOF의 calibration과 precision 비교를 수행하고, Qwen+TF-IDF stacker 및
선택적 anchor/assessment 후보를 만든 뒤 validation/test에 같은 signature로 적용한다.


## 1. 독립 Colab 부트스트랩


In [ ]:
from __future__ import annotations

import importlib.util
import json
import os
import re
import shlex
import shutil
import subprocess
import sys
from pathlib import Path

IN_COLAB = importlib.util.find_spec("google.colab") is not None
REPO_URL = "https://github.com/Chika1472/Writing-validation.git"
# 이 노트북 변환 시점의 소스 commit. 새 코드를 사용할 때만 의도적으로 바꾼다.
REPO_REF = os.environ.get(
    "WRITING_VALIDATION_REPO_REF",
    "30c70cb628208e004515144cc999e81d794bbe95",
).strip()
COLAB_PROJECT_DIR = Path("/content/Writing-validation")
CLONE_IF_MISSING = True

# 긴 학습은 두 값을 True로 두어 세션이 끊겨도 동일 절대경로를 유지한다.
USE_GOOGLE_DRIVE = False
DRIVE_ARTIFACT_DIR = Path("/content/drive/MyDrive/Writing-validation/artifacts")
DRIVE_HF_HOME = Path("/content/drive/MyDrive/Writing-validation/huggingface")


def run_process(command, *, cwd=None, env=None):
    command = [str(part) for part in command]
    print("$", shlex.join(command))
    return subprocess.run(
        command,
        cwd=str(cwd) if cwd else None,
        env=env,
        check=True,
    )


if IN_COLAB:
    if USE_GOOGLE_DRIVE:
        from google.colab import drive

        drive.mount("/content/drive")
    if not (COLAB_PROJECT_DIR / ".git").exists():
        if not CLONE_IF_MISSING:
            raise FileNotFoundError(f"저장소가 없습니다: {COLAB_PROJECT_DIR}")
        run_process(["git", "clone", REPO_URL, COLAB_PROJECT_DIR])
        run_process(["git", "checkout", "--detach", REPO_REF], cwd=COLAB_PROJECT_DIR)
    PROJECT_ROOT = COLAB_PROJECT_DIR.resolve()
else:
    candidates = [Path.cwd(), Path.cwd().parent]
    PROJECT_ROOT = next(
        (candidate.resolve() for candidate in candidates if (candidate / "pyproject.toml").is_file()),
        None,
    )
    if PROJECT_ROOT is None:
        raise FileNotFoundError("pyproject.toml이 있는 저장소 루트에서 실행하세요.")

os.chdir(PROJECT_ROOT)

if USE_GOOGLE_DRIVE:
    DRIVE_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
    artifact_link = PROJECT_ROOT / "artifacts"
    if artifact_link.exists() and not artifact_link.is_symlink():
        raise RuntimeError(
            "artifacts가 이미 실제 디렉터리입니다. 산출물을 이동/삭제하지 말고 "
            "새 RUN_NAMESPACE를 사용해 경로 계약을 유지하세요."
        )
    if not artifact_link.exists():
        artifact_link.symlink_to(DRIVE_ARTIFACT_DIR, target_is_directory=True)
    DRIVE_HF_HOME.mkdir(parents=True, exist_ok=True)
    os.environ["HF_HOME"] = str(DRIVE_HF_HOME)

# 변환 전 commit의 옛 설정 경로도 Colab Linux에서 안전하게 호환한다.
compatibility_links = [
    (PROJECT_ROOT / "데이터셋", PROJECT_ROOT / "dataset"),
    (PROJECT_ROOT / "qwen3_14b_zero_shot", PROJECT_ROOT / "result" / "qwen3_14b_zero_shot"),
]
if IN_COLAB:
    for alias, target in compatibility_links:
        if not alias.exists() and target.exists():
            alias.symlink_to(target, target_is_directory=True)


def run_cli(label: str, enabled: bool, *arguments: object, env=None):
    command = [sys.executable, *[str(argument) for argument in arguments]]
    print(f"\n[{label}]", "RUN" if enabled else "SKIP")
    print("$", shlex.join(command))
    if not enabled:
        return None
    return subprocess.run(command, cwd=PROJECT_ROOT, env=env, check=True)


def require_paths(*paths: Path) -> None:
    missing = [str(path) for path in paths if not Path(path).exists()]
    if missing:
        raise FileNotFoundError("필수 파일이 없습니다: " + ", ".join(missing))


def require_model_revision(value: str, *, enabled: bool) -> None:
    if enabled and re.fullmatch(r"[0-9a-fA-F]{40}", value or "") is None:
        raise ValueError("MODEL_REVISION에 40자리 Hugging Face commit SHA를 입력하세요.")


def require_bf16_gpu(*, enabled: bool, recommended_vram_gb: float = 22.0) -> None:
    if not enabled:
        print("GPU 단계가 꺼져 있어 BF16 검사를 건너뜁니다.")
        return
    import torch

    if not torch.cuda.is_available():
        raise RuntimeError("CUDA GPU가 필요합니다. Colab 런타임을 GPU로 변경하세요.")
    if not torch.cuda.is_bf16_supported():
        raise RuntimeError("이 파이프라인은 BF16이 필요합니다. T4 대신 L4/A100급을 선택하세요.")
    properties = torch.cuda.get_device_properties(0)
    vram_gb = properties.total_memory / 1024**3
    print({"gpu": properties.name, "vram_gb": round(vram_gb, 1), "bf16": True})
    if vram_gb < recommended_vram_gb:
        print(f"경고: 권장 VRAM {recommended_vram_gb:.0f}GB보다 작아 OOM 가능성이 큽니다.")


def show_json(path: Path):
    path = Path(path)
    if not path.exists():
        print("없음:", path)
        return None
    payload = json.loads(path.read_text(encoding="utf-8"))
    print(json.dumps(payload, ensure_ascii=False, indent=2)[:12000])
    return payload


commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True
).strip()
if IN_COLAB and commit != REPO_REF:
    raise RuntimeError(
        f"기존 Colab checkout {commit}이 고정 REPO_REF {REPO_REF}와 다릅니다. "
        "새 런타임/새 PROJECT_DIR를 사용하거나 모든 노트북에서 같은 ref를 지정하세요."
    )
print({
    "in_colab": IN_COLAB,
    "project_root": str(PROJECT_ROOT),
    "git_commit": commit,
    "artifacts": str((PROJECT_ROOT / "artifacts").resolve()),
    "hf_home": os.environ.get("HF_HOME"),
})


In [ ]:
INSTALL_DEPENDENCIES = IN_COLAB
PROJECT_EXTRAS = "qwen,test"

if INSTALL_DEPENDENCIES:
    run_process(
        [sys.executable, "-m", "pip", "install", "-q", "-e", f".[{PROJECT_EXTRAS}]"],
        cwd=PROJECT_ROOT,
    )
else:
    print("로컬 검증에서는 기존 환경을 사용합니다.")


## 2. 공통 파라미터와 실행 스위치


In [ ]:
# 실행 전에 반드시 실제 값을 채운다. moving tag나 branch 이름은 허용되지 않는다.
MODEL_REVISION = os.environ.get("MODEL_REVISION", "").strip()
ALLOW_MODEL_DOWNLOAD = False
RUN_NAMESPACE = "v1"

EXPERIMENT_ID = f"qwen3_scorer_{RUN_NAMESPACE}"
FIXED_EPOCH = 3
SEED = 42
FOLDS = range(5)

TRAIN_PATH = PROJECT_ROOT / "dataset" / "글쓰기채점능력평가2026_train.jsonl"
VALIDATION_PATH = PROJECT_ROOT / "dataset" / "글쓰기채점능력평가2026_validation.jsonl"
TEST_PATH = PROJECT_ROOT / "dataset" / "test.jsonl"  # 외부 입력
FOLDS_PATH = PROJECT_ROOT / "artifacts" / "folds" / "folds_5fold_seed42.jsonl"
QWEN_OOF = PROJECT_ROOT / "artifacts" / "predictions" / f"{EXPERIMENT_ID}_seed{SEED}_oof.jsonl"
TFIDF_OOF = PROJECT_ROOT / "artifacts" / "predictions" / "cpu_v1" / "tfidf_ridge_oof.jsonl"
TFIDF_MODEL = PROJECT_ROOT / "artifacts" / "predictions" / "cpu_v1" / "tfidf_ridge_ensemble.json"

RUN_CALIBRATION = False
RUN_BF16_OOF = False
RUN_PRECISION_COMPARISON = False
RUN_VALIDATION_QWEN = False
RUN_TWO_SOURCE_STACKER = False
RUN_VALIDATION_TFIDF = False
RUN_APPLY_STACKER = False
RUN_ANCHOR_REFERENCE_EMBEDDINGS = False
RUN_ANCHOR_OOF = False
RUN_ASSESSMENT_CACHE = False
RUN_ASSESSMENT_FIT = False
RUN_TARGET_QUERY_EMBEDDINGS = False
RUN_TARGET_ANCHOR = False
RUN_TARGET_ASSESSMENT_CACHE = False
RUN_TARGET_ASSESSMENT_PREDICTION = False
RUN_MULTISOURCE_STACKER = False
RUN_APPLY_MULTISOURCE = False
# OOF gate를 통과한 것만 추가한다: [], ["anchor"], ["assessment"], 또는 둘 다.
PROMOTED_AUXILIARY_SOURCES = []

require_model_revision(MODEL_REVISION, enabled=any([
    RUN_BF16_OOF, RUN_VALIDATION_QWEN, RUN_ANCHOR_REFERENCE_EMBEDDINGS, RUN_ASSESSMENT_CACHE,
    RUN_TARGET_QUERY_EMBEDDINGS, RUN_TARGET_ASSESSMENT_CACHE,
]))
require_bf16_gpu(enabled=any([
    RUN_BF16_OOF, RUN_VALIDATION_QWEN, RUN_ANCHOR_REFERENCE_EMBEDDINGS, RUN_ASSESSMENT_CACHE,
    RUN_TARGET_QUERY_EMBEDDINGS, RUN_TARGET_ASSESSMENT_CACHE,
]))


download_flag = ["--allow-download"] if ALLOW_MODEL_DOWNLOAD else []


In [ ]:
def checkpoint(fold: int) -> Path:
    return (
        PROJECT_ROOT / "artifacts" / "models" / EXPERIMENT_ID
        / f"seed_{SEED}__fold_{fold}" / f"epoch_{FIXED_EPOCH}"
    )

checkpoints = [checkpoint(fold) for fold in FOLDS]
CALIBRATOR = PROJECT_ROOT / "artifacts" / "calibrators" / f"{EXPERIMENT_ID}_affine.json"
CALIBRATED_OOF = PROJECT_ROOT / "artifacts" / "predictions" / f"{EXPERIMENT_ID}_crossfit_calibrated.jsonl"
QWEN_VALIDATION_RAW = PROJECT_ROOT / "artifacts" / "predictions" / f"{EXPERIMENT_ID}_validation_raw.jsonl"
QWEN_VALIDATION_CALIBRATED = PROJECT_ROOT / "artifacts" / "predictions" / f"{EXPERIMENT_ID}_validation_calibrated.jsonl"


## 3. OOF calibration


In [ ]:
run_cli(
    "Qwen OOF calibration",
    RUN_CALIBRATION,
    "scripts/fit_calibrator.py",
    "--config", "configs/data.yaml",
    "--gold", TRAIN_PATH,
    "--pred", QWEN_OOF,
    "--folds", FOLDS_PATH,
    "--output", CALIBRATOR,
    "--calibrated-output", CALIBRATED_OOF,
)


## 4. 동일 checkpoint 4bit/BF16 비교


In [ ]:
BF16_OOF = PROJECT_ROOT / "artifacts" / "predictions" / f"{EXPERIMENT_ID}_bf16_oof.jsonl"
precision_arguments: list[object] = [
    "scripts/build_precision_oof.py", "--gold", TRAIN_PATH, "--folds", FOLDS_PATH,
]
for path in checkpoints:
    precision_arguments.extend(["--checkpoint", path])
precision_arguments.extend([
    "--precision", "bf16", "--model", EXPERIMENT_ID, "--output", BF16_OOF,
])
precision_arguments.extend(download_flag)
run_cli("build BF16 OOF", RUN_BF16_OOF, *precision_arguments)

run_cli(
    "BF16 vs 4bit",
    RUN_PRECISION_COMPARISON,
    "scripts/compare_precisions.py",
    "--config", "configs/precision_comparison.yaml",
    "--gold", TRAIN_PATH,
    "--folds", FOLDS_PATH,
    "--candidate", BF16_OOF,
    "--baseline", QWEN_OOF,
    "--output", PROJECT_ROOT / "artifacts" / "reports" / f"{EXPERIMENT_ID}_bf16_vs_4bit.json",
)


## 5. validation Qwen raw/calibrated 추론


In [ ]:
def scorer_arguments(output: Path, *, calibrator: Path | None) -> list[object]:
    arguments: list[object] = ["scripts/predict_scorer.py", "--input", VALIDATION_PATH]
    for path in checkpoints:
        arguments.extend(["--checkpoint", path])
    if calibrator is not None:
        arguments.extend(["--calibrator", calibrator])
    arguments.extend([
        "--precision", "4bit", "--batch-size", "1",
        "--model-name", f"{EXPERIMENT_ID}_seed{SEED}", "--output", output,
    ])
    arguments.extend(download_flag)
    return arguments

run_cli("validation raw Qwen", RUN_VALIDATION_QWEN, *scorer_arguments(QWEN_VALIDATION_RAW, calibrator=None))
run_cli(
    "validation calibrated Qwen",
    RUN_VALIDATION_QWEN,
    *scorer_arguments(QWEN_VALIDATION_CALIBRATED, calibrator=CALIBRATOR),
)


## 6. Qwen + TF-IDF simplex stacker


In [ ]:
STACKER = PROJECT_ROOT / "artifacts" / "stackers" / f"{RUN_NAMESPACE}_qwen_tfidf.json"
STACKER_CROSSFIT = PROJECT_ROOT / "artifacts" / "predictions" / f"{RUN_NAMESPACE}_qwen_tfidf_crossfit.jsonl"
TFIDF_VALIDATION = PROJECT_ROOT / "artifacts" / "predictions" / f"{RUN_NAMESPACE}_tfidf_validation.jsonl"
STACKED_VALIDATION = PROJECT_ROOT / "artifacts" / "predictions" / f"{RUN_NAMESPACE}_stacked_validation.jsonl"

run_cli(
    "fit two-source stacker",
    RUN_TWO_SOURCE_STACKER,
    "scripts/fit_stacker.py",
    "--config", "configs/stacker.yaml",
    "--gold", TRAIN_PATH,
    "--folds", FOLDS_PATH,
    "--source", f"qwen={QWEN_OOF}",
    "--source", f"tfidf={TFIDF_OOF}",
    "--output", STACKER,
    "--crossfit-output", STACKER_CROSSFIT,
    "--report", PROJECT_ROOT / "artifacts" / "reports" / f"{RUN_NAMESPACE}_qwen_tfidf.json",
    "--model-name", f"{RUN_NAMESPACE}_qwen_tfidf",
)
run_cli(
    "validation TF-IDF",
    RUN_VALIDATION_TFIDF,
    "scripts/predict_baseline.py",
    "--input", VALIDATION_PATH,
    "--model", TFIDF_MODEL,
    "--output", TFIDF_VALIDATION,
)
run_cli(
    "apply validation stacker",
    RUN_APPLY_STACKER,
    "scripts/apply_stacker.py",
    "--stacker", STACKER,
    "--source", f"qwen={QWEN_VALIDATION_RAW}",
    "--source", f"tfidf={TFIDF_VALIDATION}",
    "--output", STACKED_VALIDATION,
)


## 7. 선택 후보: anchor/KNN

각 fold의 reference embedding은 같은 checkpoint 공간에서 만들며 held fold label을
anchor bank에서 제외한다. 아래 출력들은 Qwen+TF-IDF가 검증된 뒤에만 승격 후보로 쓴다.


In [ ]:
EMBEDDING_DIR = PROJECT_ROOT / "artifacts" / "embeddings" / RUN_NAMESPACE
ANCHOR_OOF = PROJECT_ROOT / "artifacts" / "predictions" / f"{RUN_NAMESPACE}_anchor_oof.jsonl"
ANCHOR_BANK = PROJECT_ROOT / "artifacts" / "anchors" / f"{RUN_NAMESPACE}_anchor_bank.npz"

for fold, checkpoint_path in enumerate(checkpoints):
    run_cli(
        f"reference embedding fold {fold}",
        RUN_ANCHOR_REFERENCE_EMBEDDINGS,
        "scripts/extract_embeddings.py",
        "--input", TRAIN_PATH,
        "--checkpoint", checkpoint_path,
        "--role", "reference",
        "--precision", "4bit",
        "--batch-size", "1",
        "--output", EMBEDDING_DIR / f"train_fold{fold}.npz",
        *download_flag,
    )

anchor_arguments: list[object] = [
    "scripts/build_anchor_oof.py", "--config", "configs/anchor_knn.yaml",
    "--gold", TRAIN_PATH, "--folds", FOLDS_PATH,
]
for fold in FOLDS:
    anchor_arguments.extend(["--embedding", f"{fold}={EMBEDDING_DIR / f'train_fold{fold}.npz'}"])
anchor_arguments.extend([
    "--oof-output", ANCHOR_OOF, "--anchor-bank", ANCHOR_BANK,
    "--diagnostics", PROJECT_ROOT / "artifacts" / "reports" / f"{RUN_NAMESPACE}_anchor_neighbors.jsonl",
    "--report", PROJECT_ROOT / "artifacts" / "reports" / f"{RUN_NAMESPACE}_anchor.json",
    "--model-name", f"{RUN_NAMESPACE}_anchor",
])
run_cli("build anchor OOF", RUN_ANCHOR_OOF, *anchor_arguments)


## 8. 선택 후보: assessment-question Ridge


In [ ]:
ASSESSMENT_CACHE = PROJECT_ROOT / "artifacts" / "assessment" / f"{RUN_NAMESPACE}_train_features.npz"
ASSESSMENT_DIR = PROJECT_ROOT / "artifacts" / "assessment" / f"{RUN_NAMESPACE}_ridge"

run_cli(
    "cache assessment logits",
    RUN_ASSESSMENT_CACHE,
    "scripts/cache_assessment_logits.py",
    "--config", "configs/assessment_questions.yaml",
    "--input", TRAIN_PATH,
    "--model-revision", MODEL_REVISION,
    "--tokenizer-revision", MODEL_REVISION,
    "--output", ASSESSMENT_CACHE,
    *download_flag,
)
run_cli(
    "fit assessment branch",
    RUN_ASSESSMENT_FIT,
    "scripts/fit_assessment_branch.py",
    "--config", "configs/assessment_questions.yaml",
    "--gold", TRAIN_PATH,
    "--cache", ASSESSMENT_CACHE,
    "--folds", FOLDS_PATH,
    "--output-dir", ASSESSMENT_DIR,
    "--model-name", f"{RUN_NAMESPACE}_assessment",
)


## 9. auxiliary validation 예측

OOF에서 독립 gate를 통과한 후보만 validation에 적용한다. query embedding은 reference와
같은 fold checkpoint 공간에서 만들고, assessment는 train cache와 동일한 revision·질문
계약을 사용한다.


In [ ]:
TARGET_QUERY_DIR = PROJECT_ROOT / "artifacts" / "embeddings" / f"{RUN_NAMESPACE}_validation"
ANCHOR_VALIDATION = PROJECT_ROOT / "artifacts" / "predictions" / f"{RUN_NAMESPACE}_anchor_validation.jsonl"
ASSESSMENT_VALIDATION_CACHE = (
    PROJECT_ROOT / "artifacts" / "assessment" / f"{RUN_NAMESPACE}_validation_features.npz"
)
ASSESSMENT_VALIDATION = (
    PROJECT_ROOT / "artifacts" / "predictions" / f"{RUN_NAMESPACE}_assessment_validation.jsonl"
)

for fold, checkpoint_path in enumerate(checkpoints):
    run_cli(
        f"validation query embedding fold {fold}",
        RUN_TARGET_QUERY_EMBEDDINGS,
        "scripts/extract_embeddings.py",
        "--input", VALIDATION_PATH,
        "--checkpoint", checkpoint_path,
        "--role", "query",
        "--precision", "4bit",
        "--batch-size", "1",
        "--output", TARGET_QUERY_DIR / f"validation_fold{fold}.npz",
        *download_flag,
    )

target_anchor_arguments: list[object] = [
    "scripts/predict_anchor.py", "--input", VALIDATION_PATH, "--anchor-bank", ANCHOR_BANK,
]
for fold in FOLDS:
    target_anchor_arguments.extend([
        "--embedding", f"{fold}={TARGET_QUERY_DIR / f'validation_fold{fold}.npz'}",
    ])
target_anchor_arguments.extend([
    "--output", ANCHOR_VALIDATION,
    "--diagnostics", PROJECT_ROOT / "artifacts" / "reports" / f"{RUN_NAMESPACE}_anchor_validation_neighbors.jsonl",
    "--model-name", f"{RUN_NAMESPACE}_anchor",
])
run_cli("predict validation anchor", RUN_TARGET_ANCHOR, *target_anchor_arguments)

run_cli(
    "cache validation assessment logits",
    RUN_TARGET_ASSESSMENT_CACHE,
    "scripts/cache_assessment_logits.py",
    "--config", "configs/assessment_questions.yaml",
    "--input", VALIDATION_PATH,
    "--model-revision", MODEL_REVISION,
    "--tokenizer-revision", MODEL_REVISION,
    "--output", ASSESSMENT_VALIDATION_CACHE,
    *download_flag,
)
run_cli(
    "predict validation assessment branch",
    RUN_TARGET_ASSESSMENT_PREDICTION,
    "scripts/predict_assessment_branch.py",
    "--input", VALIDATION_PATH,
    "--cache", ASSESSMENT_VALIDATION_CACHE,
    "--model", ASSESSMENT_DIR / "assessment_ridge.json",
    "--output", ASSESSMENT_VALIDATION,
)


## 10. 네 source 후보 stacker


In [ ]:
MULTISOURCE_STACKER = PROJECT_ROOT / "artifacts" / "stackers" / f"{RUN_NAMESPACE}_multisource.json"
MULTISOURCE_CROSSFIT = PROJECT_ROOT / "artifacts" / "predictions" / f"{RUN_NAMESPACE}_multisource_crossfit.jsonl"
MULTISOURCE_VALIDATION = PROJECT_ROOT / "artifacts" / "predictions" / f"{RUN_NAMESPACE}_multisource_validation.jsonl"
ASSESSMENT_OOF = ASSESSMENT_DIR / "assessment_oof.jsonl"

valid_auxiliary_sources = {"anchor", "assessment"}
unknown_sources = set(PROMOTED_AUXILIARY_SOURCES).difference(valid_auxiliary_sources)
if unknown_sources:
    raise ValueError(f"알 수 없는 auxiliary source: {sorted(unknown_sources)}")
if (RUN_MULTISOURCE_STACKER or RUN_APPLY_MULTISOURCE) and not PROMOTED_AUXILIARY_SOURCES:
    raise ValueError("auxiliary source가 없으면 앞 절의 two-source stacker를 사용하세요.")

fit_multisource_arguments: list[object] = [
    "scripts/fit_stacker.py",
    "--config", "configs/stacker.yaml",
    "--gold", TRAIN_PATH,
    "--folds", FOLDS_PATH,
    "--source", f"qwen={QWEN_OOF}",
    "--source", f"tfidf={TFIDF_OOF}",
]
oof_auxiliary_paths = {"anchor": ANCHOR_OOF, "assessment": ASSESSMENT_OOF}
for alias in PROMOTED_AUXILIARY_SOURCES:
    fit_multisource_arguments.extend(["--source", f"{alias}={oof_auxiliary_paths[alias]}"])
fit_multisource_arguments.extend([
    "--output", MULTISOURCE_STACKER,
    "--crossfit-output", MULTISOURCE_CROSSFIT,
    "--report", PROJECT_ROOT / "artifacts" / "reports" / f"{RUN_NAMESPACE}_multisource.json",
    "--model-name", f"{RUN_NAMESPACE}_multisource",
])
run_cli("fit promoted multisource stacker", RUN_MULTISOURCE_STACKER, *fit_multisource_arguments)

apply_multisource_arguments: list[object] = [
    "scripts/apply_stacker.py",
    "--stacker", MULTISOURCE_STACKER,
    "--source", f"qwen={QWEN_VALIDATION_RAW}",
    "--source", f"tfidf={TFIDF_VALIDATION}",
]
target_auxiliary_paths = {"anchor": ANCHOR_VALIDATION, "assessment": ASSESSMENT_VALIDATION}
for alias in PROMOTED_AUXILIARY_SOURCES:
    apply_multisource_arguments.extend(["--source", f"{alias}={target_auxiliary_paths[alias]}"])
apply_multisource_arguments.extend(["--output", MULTISOURCE_VALIDATION])
run_cli("apply validation multisource stacker", RUN_APPLY_MULTISOURCE, *apply_multisource_arguments)


## 11. 승격과 다음 단계

stacker에는 calibrated Qwen이 아니라 동일 raw Qwen signature를 넣는다. anchor와
assessment는 각 OOF gate 및 시간 예산을 통과한 경우에만 같은 alias로 fit/apply 양쪽에
추가한다. 확정된 genuine cross-fit score는 `05_rationale_and_finalization.ipynb`로 넘긴다.
